<a href="https://colab.research.google.com/github/Chanthul4054/E-Policing-An-Integrated-Spatio-Temporal-Crime-Forecasting-and-Decision-Support-System/blob/Pattern-Recognition-System/rule_filtering_and_pattern_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [131]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [132]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules

DATASET_PATH = "/content/pattern_recognition_preprocessed.csv"

In [133]:
TOP_K_PATTERNS = 10

MIN_SUPPORT = 0.03
MIN_CONFIDENCE = 0.30
MIN_LIFT = 1.05

MAX_RULE_LEN = 4

MIN_CONTEXT_ITEMS = 2
CONF_SIMILARITY_TOL = 0.02

OUTPUT_JSON = "final_rule_database.json"

In [134]:
df = pd.read_csv(DATASET_PATH)
print("Dataset shape:", df.shape)

required_cols = ["gn_pcode", "gn_division"]

for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Dataset missing required column: {c}")

Dataset shape: (1608, 46)


In [135]:
# AUTO DETECT FEATURE COLUMNS

crime_columns = [c for c in df.columns if c.startswith("crime_")]
location_columns = [c for c in df.columns if c.startswith("loc_")]

excluded_cols = set(["gn_pcode", "gn_division"])
excluded_cols |= set(crime_columns)
excluded_cols |= set(location_columns)

context_columns = []

In [136]:
for c in df.columns:
    if c in excluded_cols:
        continue

    if df[c].dropna().isin([0, 1, True, False]).all():
        context_columns.append(c)

selected_columns = context_columns + location_columns + crime_columns

In [137]:
print("Detected crime columns:", len(crime_columns))
print("Detected location columns:", len(location_columns))
print("Detected context columns:", len(context_columns))

Detected crime columns: 6
Detected location columns: 13
Detected context columns: 13


In [138]:
# HELPER FUNCTIONS

def context_to_crime(rule_row):
    antecedents = rule_row["antecedents"]
    consequents = rule_row["consequents"]

    ant_has_crime = any(item.startswith("crime_") for item in antecedents)
    con_has_crime = any(item.startswith("crime_") for item in consequents)

    return (not ant_has_crime) and con_has_crime

In [139]:
def get_rule_context_items(rule_row):
    ants = set(rule_row["antecedents"])
    context_items = [x for x in ants if (not x.startswith("crime_")) and (not x.startswith("loc_"))]
    return context_items

In [140]:
def contains_required_structure(rule_row):

    ctx = get_rule_context_items(rule_row)

    has_time = any(("time_" in x) for x in ctx)
    has_day = any(("day_" in x) or ("weekday" in x) or ("weekend" in x) for x in ctx)

    return has_time and has_day

In [141]:
def subset_prune_rules(rules_df, conf_tol=0.02):

    rules_df = rules_df.copy()

    keep = [True] * len(rules_df)
    rules_list = rules_df.to_dict("records")

    for i in range(len(rules_list)):

        if not keep[i]:
            continue

        A = rules_list[i]
        A_ant = set(A["antecedents"])
        A_conf = float(A["confidence"])

        for j in range(len(rules_list)):

            if i == j or not keep[j]:
                continue

            B = rules_list[j]
            B_ant = set(B["antecedents"])
            B_conf = float(B["confidence"])

            if A_ant.issubset(B_ant) and len(B_ant) > len(A_ant):

                if abs(A_conf - B_conf) <= conf_tol:
                    keep[j] = False

    return rules_df.loc[keep].copy()

In [142]:
def generate_pattern_id(rule_row, forced_location=None):
    antecedents = sorted(list(rule_row["antecedents"]))
    consequents = sorted(list(rule_row["consequents"]))

    # force include location in id
    if forced_location is not None and forced_location not in antecedents:
        antecedents.append(forced_location)
        antecedents = sorted(antecedents)

    # crime part
    crime_part = None
    for c in consequents:
        if c.startswith("crime_"):
            crime_part = c.replace("crime_", "")
            break

    if crime_part is None:
        crime_part = "unknowncrime"

    # build remaining parts
    parts = []
    for a in antecedents:
        if a.startswith("crime_"):
            continue
        if a.startswith("loc_"):
            parts.append("loc_" + a.replace("loc_", ""))
        else:
            parts.append(a)

    return crime_part + "_" + "_".join(parts)

In [143]:
def make_pattern_text(rule_row):
    ants = sorted(list(rule_row["antecedents"]))

    readable = []
    for a in ants:
        if a.startswith("loc_"):
            readable.append("Location=" + a.replace("loc_", ""))
        else:
            readable.append(a.replace("_", " "))

    return ", ".join(readable)

In [144]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
# MAIN RULE GENERATION LOOP
ALL_RULES = []

unique_gns = df[["gn_pcode", "gn_division"]].drop_duplicates().reset_index(drop=True)

for _, gn_row in unique_gns.iterrows():

    GN_PCODE = gn_row["gn_pcode"]
    GN_DIV = gn_row["gn_division"]

    df_gn = df[df["gn_pcode"] == GN_PCODE].copy()

    if df_gn.empty:
        continue

    for loc_col in location_columns:

        df_loc = df_gn[df_gn[loc_col] == 1].copy()

        if df_loc.empty:
            continue

        transaction_df = df_loc[selected_columns].fillna(0).astype(int)
        transaction_df = transaction_df.map(lambda x: True if x == 1 else False)

        frequent_itemsets = apriori(
            transaction_df,
            min_support=MIN_SUPPORT,
            use_colnames=True,
            max_len=MAX_RULE_LEN
        )

        if frequent_itemsets.empty:
            continue

        # Association rules
        rules_df = association_rules(
            frequent_itemsets,
            metric="confidence",
            min_threshold=MIN_CONFIDENCE
        )

        rules_df = rules_df.replace([np.inf, -np.inf], np.nan)
        rules_df = rules_df.dropna()

        if rules_df.empty:
            continue

        # Statistical filtering
        rules_df = rules_df[
            (rules_df["confidence"] >= MIN_CONFIDENCE) &
            (rules_df["lift"] >= MIN_LIFT)
        ].copy()

        if rules_df.empty:
            continue

        # Context -> crime only
        rules_df = rules_df[rules_df.apply(context_to_crime, axis=1)].copy()
        if rules_df.empty:
            continue

        # Must include time + day
        rules_df = rules_df[rules_df.apply(contains_required_structure, axis=1)].copy()
        if rules_df.empty:
            continue

        # Minimum context size
        rules_df["ctx_size"] = rules_df.apply(lambda r: len(get_rule_context_items(r)), axis=1)
        rules_df = rules_df[rules_df["ctx_size"] >= MIN_CONTEXT_ITEMS].copy()
        if rules_df.empty:
            continue

        # Create pattern_id
        rules_df["pattern_id"] = rules_df.apply(
            lambda r: generate_pattern_id(r, forced_location=loc_col),
            axis=1
        )

        # Create pattern_text
        rules_df["pattern_text"] = rules_df.apply(make_pattern_text, axis=1)

        # Sort
        rules_df = rules_df.sort_values(by=["confidence", "lift"], ascending=False).copy()

        # Remove duplicate patterns
        rules_df = rules_df.drop_duplicates(subset=["pattern_id"], keep="first")

        # Redundancy prune
        rules_df = subset_prune_rules(rules_df, conf_tol=CONF_SIMILARITY_TOL)

        # Now store rules separately per crime type
        for crime_col in crime_columns:

            rules_for_crime = rules_df[
                rules_df["consequents"].apply(lambda x: crime_col in x)
            ].copy()

            if rules_for_crime.empty:
                continue

            rules_for_crime = rules_for_crime.head(TOP_K_PATTERNS).copy()

            crime_name = crime_col.replace("crime_", "")
            location_name = loc_col.replace("loc_", "")

            for _, row in rules_for_crime.iterrows():
                ALL_RULES.append({
                    "gn_pcode": GN_PCODE,
                    "gn_division": GN_DIV,
                    "crime_type": crime_name,
                    "location_type": location_name,
                    "pattern_id": row["pattern_id"],
                    "pattern_text": row["pattern_text"],
                    "support": round(float(row["support"]), 4),
                    "confidence": round(float(row["confidence"]), 4),
                    "lift": round(float(row["lift"]), 4)
                })

In [145]:
# SAVE FINAL RULE DATABASE

final_rules_df = pd.DataFrame(ALL_RULES)

if final_rules_df.empty:

    print("\nNo rules were generated")

else:

    final_rules_df = final_rules_df.sort_values(
        by=["gn_pcode", "crime_type", "location_type", "confidence", "lift"],
        ascending=[True, True, True, False, False]
    ).reset_index(drop=True)

    final_rules_df.to_json(
        OUTPUT_JSON,
        index=False,
        orient="records"
    )

    print("\nRule database generated successfully")
    print("Saved file:", OUTPUT_JSON)


Rule database generated successfully
Saved file: final_rule_database.json
